In [2]:
# ============================================
# VAR Residual Diagnostics
# - Input : ./merged_var_input.csv
# - Uses  : VAR(1) on diff variables
# - Output:
#   ./diag_serial_multivariate.csv
#   ./diag_serial_ljungbox_per_eq.csv
#   ./diag_normality_multivariate.csv
#   ./diag_arch_lm_per_eq.csv
# ============================================

import pandas as pd
import numpy as np
from statsmodels.tsa.api import VAR
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch

# =============================
# Config
# =============================
DATA_PATH = "./merged_var_input.csv"
LAG = 1

# 진단용 lag (잔차 자기상관 검사에서 몇 시차까지 볼지)
# 보통 VAR lag보다 크게(예: 12, 20) 봅니다.
DIAG_LAGS = 20

OUT_SERIAL_MULTI = "./diag_serial_multivariate.csv"
OUT_SERIAL_LB = "./diag_serial_ljungbox_per_eq.csv"
OUT_NORMALITY_MULTI = "./diag_normality_multivariate.csv"
OUT_ARCH = "./diag_arch_lm_per_eq.csv"

# =============================
# 1) Load & Fit VAR
# =============================
df = pd.read_csv(DATA_PATH)
if "Date" in df.columns:
    df = df.drop(columns=["Date"])

cols = [c for c in df.columns if c.startswith("dlog_") or c.startswith("d_")]
data = df[cols].dropna()

print("사용 변수:", cols)
print("표본 수:", len(data))
print("VAR lag:", LAG)
print("DIAG_LAGS:", DIAG_LAGS)
print("-" * 60)

model = VAR(data)
res = model.fit(LAG)

# 잔차 (T x k)
u = res.resid

# =============================
# 2) (멀티) 잔차 자기상관: Portmanteau / Whiteness
# =============================
# statsmodels 버전에 따라 test_whiteness / test_serial_correlation 가 제공됩니다.
serial_rows = []
serial_obj = None
serial_method_used = None

for fn_name in ["test_whiteness", "test_serial_correlation"]:
    if hasattr(res, fn_name):
        try:
            serial_obj = getattr(res, fn_name)(nlags=DIAG_LAGS)
            serial_method_used = fn_name
            break
        except TypeError:
            # 일부 버전은 인자명이 다르거나 nlags 미지원
            try:
                serial_obj = getattr(res, fn_name)(DIAG_LAGS)
                serial_method_used = fn_name
                break
            except Exception:
                pass
        except Exception:
            pass

if serial_obj is not None:
    # 가능한 한 안전하게 핵심 요약만 저장
    serial_rows.append({
        "method": serial_method_used,
        "nlags": DIAG_LAGS,
        "statistic": getattr(serial_obj, "test_statistic", np.nan),
        "pvalue": getattr(serial_obj, "pvalue", np.nan),
        "df": getattr(serial_obj, "df", np.nan),
    })
    pd.DataFrame(serial_rows).to_csv(OUT_SERIAL_MULTI, index=False, encoding="utf-8-sig")
    print(f"[OK] multivariate serial test saved: {OUT_SERIAL_MULTI}")
    print(pd.DataFrame(serial_rows))
else:
    # 멀티 검정이 불가한 환경이면 빈 파일을 남기고, per-equation Ljung-Box로 대체
    pd.DataFrame([{
        "method": "not_available_in_this_statsmodels_version",
        "nlags": DIAG_LAGS,
        "statistic": np.nan,
        "pvalue": np.nan,
        "df": np.nan
    }]).to_csv(OUT_SERIAL_MULTI, index=False, encoding="utf-8-sig")
    print(f"[WARN] multivariate serial test not available. Saved placeholder: {OUT_SERIAL_MULTI}")

# =============================
# 3) (단변량) 잔차 자기상관: Ljung-Box per equation
# =============================
lb_rows = []
for c in cols:
    # Ljung-Box는 series별로 수행
    r = acorr_ljungbox(u[c], lags=[DIAG_LAGS], return_df=True)
    lb_rows.append({
        "equation": c,
        "lb_lag": DIAG_LAGS,
        "lb_stat": float(r["lb_stat"].iloc[0]),
        "lb_pvalue": float(r["lb_pvalue"].iloc[0]),
    })

lb_df = pd.DataFrame(lb_rows)
lb_df.to_csv(OUT_SERIAL_LB, index=False, encoding="utf-8-sig")
print(f"[OK] Ljung-Box per equation saved: {OUT_SERIAL_LB}")
print(lb_df)

# =============================
# 4) (멀티) 잔차 정규성: Jarque-Bera (multivariate)
# =============================
norm_rows = []
norm_obj = None
if hasattr(res, "test_normality"):
    norm_obj = res.test_normality()
    norm_rows.append({
        "method": "test_normality",
        "statistic": getattr(norm_obj, "test_statistic", np.nan),
        "pvalue": getattr(norm_obj, "pvalue", np.nan),
        "df": getattr(norm_obj, "df", np.nan),
    })
else:
    norm_rows.append({
        "method": "not_available_in_this_statsmodels_version",
        "statistic": np.nan,
        "pvalue": np.nan,
        "df": np.nan,
    })

pd.DataFrame(norm_rows).to_csv(OUT_NORMALITY_MULTI, index=False, encoding="utf-8-sig")
print(f"[OK] normality test saved: {OUT_NORMALITY_MULTI}")
print(pd.DataFrame(norm_rows))

# =============================
# 5) (단변량) ARCH LM test per equation
# =============================
arch_rows = []
for c in cols:
    # het_arch: returns (LM stat, LM pvalue, F stat, F pvalue)
    lm_stat, lm_p, f_stat, f_p = het_arch(u[c], nlags=DIAG_LAGS)
    arch_rows.append({
        "equation": c,
        "arch_lags": DIAG_LAGS,
        "lm_stat": float(lm_stat),
        "lm_pvalue": float(lm_p),
        "f_stat": float(f_stat),
        "f_pvalue": float(f_p),
    })

arch_df = pd.DataFrame(arch_rows)
arch_df.to_csv(OUT_ARCH, index=False, encoding="utf-8-sig")
print(f"[OK] ARCH LM per equation saved: {OUT_ARCH}")
print(arch_df)

print("-" * 60)
print("완료. 다음은 진단 결과 해석(통과/미통과에 따른 조치) 단계로 넘어가면 됩니다.")

사용 변수: ['dlog_SOLVPN', 'dlog_COPPER', 'dlog_DXY', 'd_UST10Y', 'dlog_VIX']
표본 수: 1325
VAR lag: 1
DIAG_LAGS: 20
------------------------------------------------------------
[OK] multivariate serial test saved: ./diag_serial_multivariate.csv
           method  nlags   statistic    pvalue   df
0  test_whiteness     20  500.680105  0.200412  475
[OK] Ljung-Box per equation saved: ./diag_serial_ljungbox_per_eq.csv
      equation  lb_lag    lb_stat  lb_pvalue
0  dlog_SOLVPN      20  17.394290   0.627242
1  dlog_COPPER      20  34.766575   0.021383
2     dlog_DXY      20  26.440332   0.151757
3     d_UST10Y      20  20.152242   0.448443
4     dlog_VIX      20  22.034129   0.338662
[OK] normality test saved: ./diag_normality_multivariate.csv
           method     statistic  pvalue  df
0  test_normality  93256.086807     0.0  10
[OK] ARCH LM per equation saved: ./diag_arch_lm_per_eq.csv
      equation  arch_lags     lm_stat     lm_pvalue    f_stat      f_pvalue
0  dlog_SOLVPN         20   48.889